# 云边联邦训练：风险估计可靠性
## Cloud-edge federated RL: reliability of risk estimates

唯一全局调参入口。先完成 `core.py` 的研究者算法，再启用真实训练与攻击。当前默认只执行版本核验和真实 Adult 数据准备，不生成模拟训练结果。

**问题**：调度诱导的训练分布变化是否造成风险低估？不确定性和有限预算审计能否改善决策？

工程说明见 README；任务顺序与研究约束见 docs。客户端编号在代码中从 0 开始。

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display

from cloud_edge_rl.notebook import execute_job
from cloud_edge_rl.pipeline import runtime_info

PROJECT_ROOT = Path.cwd()
RESULTS = {}
display(runtime_info())

## 1. 集中配置 / Central configuration

全部实验数值、变体、种子、阶段开关、输入产物与云资源只在下一单元修改。模型/算法参数是供您实现核心时使用的起点，不是已验证最优值。资源功率为模拟假设，非硬件实测。

用户已在本轮明确取消金额上限；20/100 美元仅保留为历史约束，历史费用与交互不重置。用户将全局配置上限从八组修订为九组，原八组超额记录保留；每方法/种子/划分 3000 次累计真实交互及数据隔离仍有效。未来云任务请求单 H100、16 CPU、64 GiB，保持串行；必须先通过本轮样本门并由主代理批准有界阶段。官方测试封存至方法、参考、攻击协议和配置全部冻结，不用于调参。`estimated_seconds_by_batch` 从 benchmark 产物读取，不填虚构耗时。恢复显式引用 checkpoint；改 tag 不重置交互，同名失败结果不覆盖。

In [ ]:
# 唯一调参区 / The ONLY place to edit experiment values.
# 默认只执行本地版本检查和真实数据准备；学习与攻击核心仍由您填写。
CONFIG = {
    "experiment": {"tag": "pilot-001", "variant": "setup"},
    "data": {
        "url": "https://archive.ics.uci.edu/static/public/2/adult.zip",
        "validation_fraction": 0.2,
        "split_seed": 42,
        "partition_seed": 73,
        "num_clients": 3,
        "partition": "iid",
        "label_concentration": 0.5,
    },
    "model": {
        "hidden_sizes": [64, 64],
        "activation": "relu",
        "output_size": 1,
        "loss": "binary_cross_entropy_with_logits",
        "learning_rate": 0.01,
        "optimizer": "sgd",
        "device": "cuda",
        "dtype": "float32",
    },
    "environment": {
        "client_sets": [[0, 1], [0, 2], [1, 2], [0, 1, 2]],
        "batch_sizes": [8, 32, 64],
        "max_rounds": 30,
        "time_budget_seconds": 120.0,
        "sample_budget": 3000,
        "target_validation_loss": 0.3,
        "utility_prior": [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
        "utility_ema_decay": 0.8,
    },
    "resources": {
        "interval_seconds": 2.0,
        "bandwidth_mbps": [5.0, 10.0, 20.0],
        "download_mbps": [20.0, 40.0, 60.0],
        "samples_per_second": [400.0, 800.0, 1200.0],
        "compute_watts": [5.0, 8.0, 12.0],
        "transmit_watts": [1.0, 1.5, 2.0],
        "receive_watts": [0.5, 0.7, 1.0],
        "bandwidth_jitter": 0.3,
        "load_range": [0.0, 0.6],
        "aggregation_seconds": 0.005,
        "server_watts": 30.0,
    },
    "reward": {
        "time_weight": 0.05,
        "energy_weight": 0.01,
        "risk_weight": 0.0,
        "uncertainty_weight": 0.0,
        "time_scale": 1.0,
        "energy_scale": 10.0,
    },
    "baseline": {"fixed_action": 4, "round_robin_actions": [1, 4, 7]},
    "risk": {
        "mode": "unavailable",
        "ensemble_size": 5,
        "hidden_sizes": [32, 32],
        "learning_rate": 0.001,
        "epochs": 20,
        "batch_size": 64,
        "device": "cuda",
        "validation_group_fraction": 0.2,
    },
    "rl": {
        "algorithm": "dqn",
        "device": "cuda",
        "hidden_sizes": [128, 128],
        "total_steps": 3000,
        "learning_rate": 0.0003,
        "gamma": 0.99,
        "batch_size": 64,
        "buffer_size": 20000,
        "learning_starts": 256,
        "train_frequency": 1,
        "target_update_interval": 100,
        "epsilon_start": 1.0,
        "epsilon_end": 0.05,
        "epsilon_decay_steps": 2000,
        "checkpoint_interval": 500,
        "ppo": {
            "rollout_steps": 128,
            "epochs": 4,
            "clip_ratio": 0.2,
            "gae_lambda": 0.95,
            "entropy_weight": 0.01,
        },
    },
    "attack": {
        "method": "gradient_matching",
        "iterations": 200,
        "restarts": 1,
        "learning_rate": 0.1,
        "learning_rate_schedule": "linear",
        "learning_rate_end_factor": 0.1,
        "labels_known": True,
        "device": "cuda",
    },
    "audit": {
        "policy": "periodic",
        "max_calls": 6,
        "max_seconds": 600.0,
        "period": 5,
        "ranking_floor": 0.01,
        "numeric_tolerance": 0.05,
        "device": "cuda",
        "estimated_seconds_by_batch": None,
    },
    "execution": {
        "split": "train",
        "save_updates": True,
        "warmup_actions": [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11],
    },
    "seeds": {
        "pilot": [7],
        "train": [101, 102, 103, 104, 105],
        "validation": [201, 202],
        "test": [301, 302, 303, 304, 305],
    },
    "modal": {
        "app_name": "cloud-edge-rl",
        "volume_name": "cloud-edge-rl-artifacts",
        "gpu": "H100",
        "cpu": 16.0,
        "memory_mb": 65536,
        "timeout_seconds": 1800,
        "max_containers": 1,
        "scaledown_window": 2,
        "persistent_compiler_cache": True,
        "budget_usd": None,
        "detach": True,
    },
    "variants": {
        "setup": {
            "method": "fixed",
            "risk_mode": "unavailable",
            "risk_weight": 0.0,
            "uncertainty_weight": 0.0,
        },
        "round-robin": {
            "method": "round_robin",
            "risk_mode": "unavailable",
            "risk_weight": 0.0,
            "uncertainty_weight": 0.0,
        },
        "fixed": {
            "method": "fixed",
            "risk_mode": "fitted",
            "risk_weight": 0.1,
            "uncertainty_weight": 0.0,
        },
        "time-greedy": {
            "method": "time_greedy",
            "risk_mode": "fitted",
            "risk_weight": 0.1,
            "uncertainty_weight": 0.0,
        },
        "risk-greedy": {
            "method": "risk_greedy",
            "risk_mode": "fitted",
            "risk_weight": 0.1,
            "uncertainty_weight": 0.0,
        },
        "conservative-greedy": {
            "method": "risk_greedy",
            "risk_mode": "fitted",
            "risk_weight": 0.1,
            "uncertainty_weight": 1.0,
        },
        "dqn-no-privacy": {
            "method": "dqn",
            "risk_mode": "fitted",
            "risk_weight": 0.0,
            "uncertainty_weight": 0.0,
        },
        "dqn": {
            "method": "dqn",
            "risk_mode": "fitted",
            "risk_weight": 0.1,
            "uncertainty_weight": 0.0,
        },
        "conservative-dqn": {
            "method": "dqn",
            "risk_mode": "fitted",
            "risk_weight": 0.1,
            "uncertainty_weight": 1.0,
        },
        "ppo": {
            "method": "ppo",
            "risk_mode": "fitted",
            "risk_weight": 0.1,
            "uncertainty_weight": 0.0,
        },
    },
    "tuning": {
        "learning_rate": [0.0001, 0.0003, 0.001],
        "risk_weight": [0.03, 0.1, 0.3],
        "uncertainty_weight": [0.0, 0.5, 1.0],
        "trials": 6,
        "seed": 9001,
    },
    "analysis": {
        "run_ids": [],
        "audit_run_ids": [],
        "reference_method": "risk-greedy",
        "metric": "validation_loss",
        "x_axis": "elapsed_seconds",
        "risk_threshold": 0.5,
    },
    "workflow": {
        "reuse_completed": True,
        "jobs": {
            "runtime": {
                "enabled": True,
                "stage": "runtime_check",
                "backend": "local",
                "variants": ["setup"],
                "seed_group": "pilot",
                "risk_version": "unavailable",
                "split": "train",
                "inputs": {},
            },
            "data": {
                "enabled": True,
                "stage": "prepare_data",
                "backend": "local",
                "variants": ["setup"],
                "seed_group": "pilot",
                "risk_version": "unavailable",
                "split": "train",
                "inputs": {},
            },
            "modal_check": {
                "enabled": False,
                "stage": "runtime_check",
                "backend": "modal",
                "variants": ["setup"],
                "seed_group": "pilot",
                "risk_version": "unavailable",
                "split": "train",
                "inputs": {},
            },
            "collect": {
                "enabled": False,
                "stage": "collect",
                "backend": "modal",
                "variants": ["round-robin"],
                "seed_group": "pilot",
                "risk_version": "unavailable",
                "split": "train",
                "inputs": {},
            },
            "benchmark": {
                "enabled": False,
                "stage": "benchmark_attack",
                "backend": "modal",
                "variants": ["setup"],
                "seed_group": "pilot",
                "risk_version": "unavailable",
                "split": "train",
                "inputs": {"source_run_ids": []},
            },
            "audit": {
                "enabled": False,
                "stage": "audit",
                "backend": "modal",
                "variants": ["setup"],
                "seed_group": "pilot",
                "risk_version": "unavailable",
                "split": "train",
                "inputs": {"source_run_ids": [], "benchmark_run_id": None},
            },
            "fit_risk": {
                "enabled": False,
                "stage": "fit_risk",
                "backend": "modal",
                "variants": ["setup"],
                "seed_group": "pilot",
                "risk_version": "risk-v1",
                "split": "train",
                "inputs": {"audit_run_ids": []},
            },
            "train": {
                "enabled": False,
                "stage": "train_policy",
                "backend": "modal",
                "variants": ["dqn"],
                "seed_group": "pilot",
                "risk_version": "risk-v1",
                "split": "train",
                "inputs": {"risk_checkpoint": None, "previous_training_checkpoint": None},
            },
            "tune": {
                "enabled": False,
                "stage": "train_policy",
                "backend": "modal",
                "variants": ["dqn"],
                "seed_group": "pilot",
                "risk_version": "risk-v1",
                "split": "train",
                "inputs": {"risk_checkpoint": None, "previous_training_checkpoint": None},
            },
            "validate": {
                "enabled": False,
                "stage": "validate",
                "backend": "modal",
                "variants": ["dqn"],
                "seed_group": "validation",
                "risk_version": "risk-v1",
                "split": "validation",
                "inputs": {"training_run_ids": []},
            },
            "calibrate": {
                "enabled": False,
                "stage": "calibrate_risk",
                "backend": "modal",
                "variants": ["setup"],
                "seed_group": "pilot",
                "risk_version": "risk-v2",
                "split": "train",
                "inputs": {"audit_run_ids": []},
            },
            "evaluate": {
                "enabled": False,
                "stage": "evaluate",
                "backend": "modal",
                "variants": ["dqn"],
                "seed_group": "test",
                "risk_version": "risk-v1",
                "split": "test",
                "inputs": {"training_run_ids": []},
            },
            "evaluate_audits": {
                "enabled": False,
                "stage": "evaluate_audits",
                "backend": "modal",
                "variants": ["setup"],
                "seed_group": "test",
                "risk_version": "risk-v1",
                "split": "test",
                "inputs": {"source_run_ids": [], "benchmark_run_id": None},
            },
            "evaluate_baselines": {
                "enabled": False,
                "stage": "evaluate",
                "backend": "modal",
                "variants": ["fixed", "time-greedy", "risk-greedy", "conservative-greedy"],
                "seed_group": "test",
                "risk_version": "risk-v1",
                "split": "test",
                "inputs": {"risk_checkpoint": None},
            },
        },
    },
}

# 设备是运行资源选择；全模块使用同一显式选择。 / Share the selected compute device.
for section in ("attack", "audit", "risk", "rl"):
    CONFIG[section]["device"] = CONFIG["model"]["device"]
# 外部候选独立筛选配置 / Isolated screening of external candidates.
CONFIG["reference_benchmarks"] = {
    "tag": "sota-screen-20261005",
    "enabled": {
        "tabular_gia": False,
        "tabm": False,
        "conformalopt": False,
        "metacs_fl": False,
        "pqn": False,
    },
    "tabular_gia": {
        "seeds": [7, 17, 27],
        "batch_sizes": [1, 8, 32, 64],
        "training_steps": [0, 20],
        "training_batch_size": 64,
        "training_learning_rate": 0.01,
        "attack_iterations": 1000,
        "attack_learning_rate": 0.1,
        "restarts": 1,
        "threads": 2,
        "device": "cpu",
        "hidden_size": 64,
        "hidden_layers": 2,
        "activation": "relu",
        "normalization": "none",
        "dropout": 0.0,
        "numeric_tolerance": 0.2,
        "evaluation_seed_offset": 1729,
        "tv_reg": 0.0,
        "epochs": 1,
        "label_known": True,
        "median_pooling": False,
        "top10norms": False,
    },
    "tabm": {
        "seeds": [7, 17, 27],
        "train_rows": 4096,
        "validation_rows": 1024,
        "epochs": 15,
        "batch_size": 256,
        "learning_rate": 0.001,
        "weight_decay": 0.0001,
        "n_blocks": 2,
        "d_block": 64,
        "dropout": 0.0,
        "k": 8,
        "threads": 2,
        "device": "cpu",
        "classification_threshold": 0.5,
    },
    "conformalopt": {
        "seed": 7,
        "source": "official_googl",
        "source_data_name": "GOOGL_theta_absolute-residual_scores",
        "warmup_scores": 30,
        "score_scale": 10000.0,
        "validation_fraction": 0.3,
        "test_break_fraction": 0.5,
        "alpha": 0.1,
        "quantile_method": "higher",
        "tracker_order": 1,
        "tracker_bias": 1.0,
        "learning_rate": 0.05,
        "learning_rate_type": "fixed",
        "feedback_delay": 0,
        "clip_low": 0.0,
        "clip_high": 1.0,
    },
    "metacs_fl": {
        "seeds": [7, 17, 27],
        "cases_per_seed": 5,
        "num_clients": 3,
        "total_tasks": 12,
        "capacities": [
            [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12],
            [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12],
            [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12],
        ],
        "time_per_task_range": [0.2, 1.5],
        "startup_seconds_range": [0.1, 0.5],
        "power_watts_range": [1.0, 8.0],
        "idle_watts_range": [0.01, 0.1],
        "battery_joules": 1000.0,
        "class_count": 2,
        "class_capacity": 24,
        "class_dirichlet_alpha": 0.5,
        "utility_range": [0.0, 1.0],
        "history_rounds": 5,
        "history_selection_probability": 0.6,
        "utility_alpha": 0.5,
        "class_beta": 0.5,
        "deadline_seconds": 20.0,
        "iterations": 100,
        "destroy": {"sf": 0.67, "rf_min": 0.2, "rf_max": 0.8},
        "accept": {"D_pc_min": -100.0, "M_pc_max": 10000.0, "E_pc_max": 10000.0, "bl_min": 0.0},
        "weights": {
            "M_weight": 0.3,
            "E_weight": 0.3,
            "D_weight": 0.1,
            "K_weight": 0.1,
            "U_weight": 0.2,
        },
        "normalization_bounds": {"min_M_X": 0.1, "max_M_X": 20.0, "min_E_X": 0.1, "max_E_X": 500.0},
        "class_distribution": "locally_balanced",
        "optimality_tolerance": 1e-09,
    },
    "pqn": {
        "seeds": [7, 17, 27],
        "num_eval_episodes": 32,
        "max_eval_steps": 500,
        "eval_seed_offset": 1729,
        "upstream": {
            "ALG_NAME": "pqn",
            "ENV_NAME": "CartPole-v1",
            "TOTAL_TIMESTEPS": 32768,
            "TOTAL_TIMESTEPS_DECAY": 32768,
            "NUM_ENVS": 16,
            "NUM_STEPS": 32,
            "NUM_MINIBATCHES": 4,
            "NUM_EPOCHS": 2,
            "HIDDEN_SIZE": 64,
            "NUM_LAYERS": 2,
            "NORM_TYPE": "layer_norm",
            "NORM_INPUT": False,
            "LR": 0.0003,
            "MAX_GRAD_NORM": 10.0,
            "LR_LINEAR_DECAY": True,
            "REW_SCALE": 1.0,
            "GAMMA": 0.99,
            "LAMBDA": 0.95,
            "EPS_START": 1.0,
            "EPS_FINISH": 0.05,
            "EPS_DECAY": 0.5,
            "TEST_DURING_TRAINING": True,
            "TEST_INTERVAL": 0.25,
            "TEST_NUM_ENVS": 16,
            "TEST_NUM_STEPS": 500,
            "EPS_TEST": 0.0,
            "WANDB_MODE": "disabled",
            "WANDB_LOG_ALL_SEEDS": False,
        },
    },
}

# 固定筛选数据与上游结构参数 / Fixed screening data and upstream architecture choices.
CONFIG["reference_benchmarks"]["tabm"].update(
    {
        "subset_seed": 42,
        "arch_type": "tabm",
        "activation": "ReLU",
        "start_scaling_init": "random-signs",
        "risk_kappa": 1.0,
    }
)
CONFIG["reference_benchmarks"]["conformalopt"]["scorecaster"] = None

# 官方 CartPole 训练预设另作足量筛选 / Also test the upstream CartPole training preset.
CONFIG["reference_benchmarks"]["enabled"]["pqn_paper"] = False
CONFIG["reference_benchmarks"]["pqn_paper"] = {
    **CONFIG["reference_benchmarks"]["pqn"],
    "upstream": {
        **CONFIG["reference_benchmarks"]["pqn"]["upstream"],
        "TOTAL_TIMESTEPS": 500000,
        "TOTAL_TIMESTEPS_DECAY": 500000,
        "NUM_ENVS": 32,
        "NUM_STEPS": 64,
        "NUM_MINIBATCHES": 16,
        "NUM_EPOCHS": 4,
        "HIDDEN_SIZE": 256,
        "LR": 0.0001,
        "REW_SCALE": 0.1,
        "EPS_FINISH": 0.2,
        "EPS_DECAY": 0.2,
        "TEST_INTERVAL": 0.05,
        "TEST_NUM_ENVS": 128,
        "ENV_KWARGS": {},
    },
}

# OCP 原论文反馈协议；不是稀疏审计保证 / Native OCP feedback protocol, not sparse-audit guarantees.
CONFIG["reference_benchmarks"]["enabled"]["ocp_unlock"] = False
CONFIG["reference_benchmarks"]["ocp_unlock"] = {
    "seeds": [7, 17, 27],
    "score_archive": "reference_algorithms/results/sota-screen-20261005/conformalopt/official_googl-scores.npz",
    "alpha": 0.1,
    "arms": 20,
    "calibration_upper_multiplier": 2.5,
    "score_lower_bound": 0.0,
    "lambda": 1.0,
    "delta": 1.0,
    "function": "lin",
    "displeasure": "abs",
    "loss_type": "new",
    "C": 20.0,
    "version": 0,
    "ot_power": 0.25,
    "power_const_option": 10.0,
    "variants": {"OCP-Bandit": 0, "OCP-Unlock": 3, "OCP-Unlock+": 4},
    "test_break_fraction": 0.5,
    "quantile_method": "higher",
}

# 校准学习率敏感性 / Calibration learning-rate sensitivity.
CONFIG["reference_benchmarks"]["enabled"]["conformalopt_small_lr"] = False
CONFIG["reference_benchmarks"]["conformalopt_small_lr"] = {
    **CONFIG["reference_benchmarks"]["conformalopt"],
    "learning_rate": 0.0005,
}
CONFIG["reference_benchmarks"]["ocp_unlock"]["warmup_bandit"] = True

# 同配置云端补充测评 / Cloud supplement with the same benchmark settings.
CONFIG["reference_benchmarks"]["enabled"]["tabm_modal"] = False
CONFIG["reference_benchmarks"]["tabm_modal"] = {
    **CONFIG["reference_benchmarks"]["tabm"],
    "device": "cuda",
}

# 主项目第三方对照；研究者核心仍独立 / Main-project comparisons keep researcher cores separate.
CONFIG["components"] = {"backend": "researcher", "risk": "researcher", "attack": "researcher"}
CONFIG["model"].update(
    {
        "normalization": "none",
        "dropout": 0.0,
        "binary_pos_weight": None,
        "momentum": 0.0,
        "weight_decay": 0.0,
        "evaluation_batch_size": 1024,
        "classification_threshold": 0.5,
    }
)
CONFIG["environment"]["history_window"] = 5
# 原生第一版：有限任务、暴露风险和显式训练参数 / Native finite-task protocol.
CONFIG["reward"].update({"protocol": "exposure_v1", "accuracy_weight": 1.0})
CONFIG["risk"].update({"validation_fraction": 0.2, "weight_decay": 0.0001})
CONFIG["rl"].update({"gamma": 1.0, "max_grad_norm": 10.0, "double_dqn": True,
    "rollout_steps": 128, "ppo_epochs": 4, "clip_range": 0.2, "gae_lambda": 0.95,
    "entropy_coefficient": 0.01, "value_coefficient": 0.5, "normalize_advantage": True})
CONFIG["audit"]["exclude_warmup"] = True
CONFIG["audit"].update({"max_calls": 24, "max_seconds": 120.0, "period": 2})
CONFIG["comparisons"] = {
    "modal_source_root": "/opt/cloud-edge-study",
    "risk": {
        "threads": 2,
        "epochs": 20,
        "batch_size": 16,
        "learning_rate": 0.001,
        "weight_decay": 0.0001,
        "n_blocks": 2,
        "d_block": 64,
        "dropout": 0.0,
        "k": 8,
        "arch_type": "tabm",
        "activation": "ReLU",
        "start_scaling_init": "random-signs",
        "validation_fraction": 0.5,
        "output_transform": "sigmoid",
    },
    "attack": {"tv_reg": 0.0, "epochs": 1, "median_pooling": False, "top10norms": False},
    "metacs": {
        "history_window": CONFIG["environment"]["history_window"],
        "utility_scale": 0.01,
        "weights": {
            "M_weight": 0.3,
            "E_weight": 0.3,
            "D_weight": 0.2,
            "K_weight": 0.0,
            "U_weight": 0.2,
        },
        "normalization_bounds": {
            "min_M_X": 0.001,
            "max_M_X": 20.0,
            "min_E_X": 0.01,
            "max_E_X": 500.0,
        },
    },
    "pqn": {
        "device": CONFIG["model"]["device"],
        "hidden_width": 128,
        "num_layers": 2,
        "layer_norm_epsilon": 1e-5,
        "learning_rate": 0.0001,
        "adam_betas": [0.9, 0.999],
        "adam_epsilon": 1e-8,
        "weight_decay": 0.0,
        "gamma": 0.99,
        "lambda": 0.95,
        "reward_scale": 0.1,
        "total_steps": 64,
        "rollout_steps": 16,
        "num_minibatches": 4,
        "epochs": 4,
        "epsilon_start": 1.0,
        "epsilon_finish": 0.2,
        "epsilon_decay_steps": 64,
        "linear_lr_decay": True,
        "lr_decay_steps": 64,
        "min_learning_rate": 0.0,
        "max_grad_norm": 10.0,
        "checkpoint_interval_updates": 1,
    },
    "calibration": {
        "calibration_run_ids": [],
        "evaluation_run_ids": [],
        "methods": [
            "static_quantile",
            "conformal_sqt",
            "conformal_lqt",
            "ocp_bandit",
            "ocp_unlock",
            "ocp_unlock_plus",
        ],
        "alpha": 0.1,
        "quantile_method": "higher",
        "conformal": {
            "learning_rate": 0.05,
            "learning_rate_type": "fixed",
            "tracker_order": 1,
            "tracker_bias": 1.0,
            "scorecaster": None,
            "feedback_delay": 0,
        },
        # 2 arms 用于小规模真实审计；正式比较可在足够样本后设20 / Two arms for the real small pilot; use20 with sufficient audited samples.
        "ocp": {
            "seed": 7,
            "arms": 2,
            "score_upper_bound": 1.0,
            "lambda": 1.0,
            "delta": 1.0,
            "function": "lin",
            "displeasure": "abs",
            "loss_type": "new",
            # 上游保留参数；损失实际固定为 40。 / Legacy argument; pinned loss uses 40.
            "C": 20.0,
            "version": 0,
            "ot_power": 0.25,
            "power_const_option": 10.0,
            "warmup_bandit": True,
            "variants": {"ocp_bandit": 0, "ocp_unlock": 3, "ocp_unlock_plus": 4},
        },
    },
    "workflow": {
        "enabled": False,
        "tag": "integrated-baselines-20261005-v3",
        "backend": "modal",
        "resume_manifest": "",
        "policy_seed_pairs": [{"train_seed": 7, "evaluation_seed": 1007}],
        "job_order": [
            "comparison_collect",
            "comparison_benchmark",
            "comparison_audit",
            "comparison_fit_risk",
            "comparison_train",
            "comparison_validate_collect",
            "comparison_validate_audit",
            "comparison_test_collect",
            "comparison_test_audit",
            "comparison_evaluate_risk",
            "comparison_evaluate_schedulers",
            "comparison_scheduler_audits",
            "comparison_calibration",
        ],
    },
}
CONFIG["seeds"].update(
    {
        "comparison_train": [7, 17],
        "comparison_policy_train": [7],
        "comparison_validation": [107],
        "comparison_test": [1007],
    }
)

# 基线选择仅改变声明的组件 / Baselines change only their declared components.
reference_components = {"backend": "tabular_mlp", "risk": "tabm", "attack": "tabular_gia"}
for name, method, fitted, risk_weight, uncertainty_weight, risk_kind in [
    ("reference-setup", "fixed", False, 0.0, 0.0, "tabm"),
    ("reference-setup-mlp", "fixed", False, 0.0, 0.0, "mlp"),
    ("reference-round-robin", "round_robin", False, 0.0, 0.0, "tabm"),
    ("reference-fixed", "fixed", True, 0.1, 0.0, "tabm"),
    ("reference-time-greedy", "time_greedy", True, 0.1, 0.0, "tabm"),
    ("reference-risk-greedy", "risk_greedy", True, 0.1, 0.0, "tabm"),
    ("reference-conservative-greedy", "risk_greedy", True, 0.1, 1.0, "tabm"),
    ("reference-risk-mlp", "fixed", True, 0.1, 0.0, "mlp"),
    ("metacs", "metacs", True, 0.1, 0.0, "tabm"),
    ("pqn", "pqn", True, 0.1, 0.0, "tabm"),
]:
    CONFIG["variants"][name] = {
        "method": method,
        "risk_mode": "fitted" if fitted else "unavailable",
        "risk_weight": risk_weight,
        "uncertainty_weight": uncertainty_weight,
        "components": {**reference_components, "risk": risk_kind},
    }

# 全局比较开关按依赖顺序运行；输入运行ID由已完成阶段生成 / The global comparison switch runs stages in dependency order.
for name, stage, variants, seed_group, split, inputs in [
    ("comparison_collect", "collect", ["reference-round-robin"], "comparison_train", "train", {}),
    (
        "comparison_benchmark",
        "benchmark_attack",
        ["reference-setup"],
        "pilot",
        "train",
        {"source_run_ids": []},
    ),
    (
        "comparison_audit",
        "audit",
        ["reference-setup"],
        "pilot",
        "train",
        {"source_run_ids": [], "benchmark_run_id": None},
    ),
    (
        "comparison_fit_risk",
        "fit_risk",
        ["reference-setup", "reference-setup-mlp"],
        "pilot",
        "train",
        {"audit_run_ids": []},
    ),
    (
        "comparison_train",
        "train_policy",
        ["pqn"],
        "comparison_policy_train",
        "train",
        {"risk_checkpoint": None, "previous_training_checkpoint": None},
    ),
    (
        "comparison_validate_collect",
        "collect",
        ["reference-fixed"],
        "comparison_validation",
        "validation",
        {"risk_checkpoint": None},
    ),
    (
        "comparison_validate_audit",
        "audit",
        ["reference-setup"],
        "pilot",
        "validation",
        {"source_run_ids": [], "benchmark_run_id": None},
    ),
    (
        "comparison_test_collect",
        "collect",
        ["reference-fixed"],
        "comparison_test",
        "test",
        {"risk_checkpoint": None},
    ),
    (
        "comparison_test_audit",
        "evaluate_audits",
        ["reference-setup"],
        "pilot",
        "test",
        {"source_run_ids": [], "benchmark_run_id": None},
    ),
    (
        "comparison_evaluate_risk",
        "evaluate_risk",
        ["reference-fixed", "reference-risk-mlp"],
        "pilot",
        "test",
        {"audit_run_ids": [], "risk_checkpoint": None},
    ),
    (
        "comparison_evaluate_schedulers",
        "evaluate",
        [
            "reference-fixed",
            "reference-time-greedy",
            "reference-risk-greedy",
            "reference-conservative-greedy",
            "metacs",
            "pqn",
        ],
        "comparison_test",
        "test",
        {"risk_checkpoint": None, "agent_checkpoint": None},
    ),
    (
        "comparison_scheduler_audits",
        "evaluate_audits",
        [
            "reference-fixed",
            "reference-time-greedy",
            "reference-risk-greedy",
            "reference-conservative-greedy",
            "metacs",
            "pqn",
        ],
        "comparison_test",
        "test",
        {"source_run_ids": [], "benchmark_run_id": None},
    ),
    (
        "comparison_calibration",
        "compare_calibration",
        ["reference-setup"],
        "pilot",
        "test",
        {"audit_run_ids": []},
    ),
]:
    CONFIG["workflow"]["jobs"][name] = {
        "enabled": False,
        "stage": stage,
        "backend": CONFIG["comparisons"]["workflow"]["backend"],
        "variants": variants,
        "seed_group": seed_group,
        "risk_version": "reference-risk-v1",
        "split": split,
        "inputs": inputs,
    }

# Modal 真实性能验收：先样本门，再扩展；默认不提交付费任务。 / Sample gate before paid scaling.
CONFIG["native_acceptance"] = {
    "enabled": False, "tag": "first-acceptance-v1-20261009",
    "version": "first-acceptance-v1",
    "artifact_verification": "json_snapshot_no_hash",
    "required_paired_seeds": 3, "minimum_feasible_improving_seeds": 2,
    "risk_multiplier": 0.95, "proxy_mae_multiplier": 1.10,
    "calibration_underestimation_multiplier": 0.95, "calibration_mae_multiplier": 1.10,
    "interaction_limit": 3000, "require_complete_audit": True,
    "confirmation": {
        "status": "registered_only", "frozen": False,
        "freeze_id": None,
        "main_method": None, "scheduler_reference": None, "risk_reference": None,
        "training_seeds": [1101, 1102, 1103],
        "evaluation_seeds": {"1101": 2101, "1102": 2102, "1103": 2103},
        "risk_fit_seeds": [3101, 3102, 3103], "panel_seeds": [4101, 4102, 4103],
        "dataset": "adult", "partition": "iid", "resources": "normal",
        "freeze_manifest": None, "evidence_manifest": None,
    },
    "reference_selection": {
        "scheduler_candidates": ["fixed", "time_greedy", "risk_greedy", "metacs", "pqn"],
        "scheduler_rule": "max_validation_accuracy_then_min_time_then_min_energy_then_min_audited_risk",
        "risk_candidates": ["mlp", "tabm"], "risk_rule": "min_development_panel_exposure_weighted_mae",
        "main_rule": "min_audited_validation_risk_among_feasible_dqn_ppo",
    },
    "research_protocol": {
        "path": "C:/Users/42155/Downloads/PLAN (1).md",
        "verification": "direct_read_no_hash",
    },
    "development": {
        "hypothesis_id": "R1-raw-vs-batch", "diagnostic_only": True,
        "status": "existing_artifacts_reviewed_development_gates_pass_formal_insufficient",
        "aggregate_manifest": {
            "path": "output/first-acceptance-v1-20261009/resume-20261009-R1-aggregate/aggregate.json",
            "role": "historical_development_only",
        },
        "risk_fit_seeds": [501, 502, 503], "panel_seeds": [21, 22],
        "change": "same_network_raw_vs_existing_batch_calibration",
        "allow_new_fedsgd_interactions": False,
        "risk_candidate": "width64-100-b16-batch-calibrated", "device": "cpu", "threads": 2,
        "output_root": "output/first-acceptance-v1-20261009/risk-R1-raw-vs-batch-cpu-v1",
        "training_sources": [
            "output/native-acceptance-20261008/common-panel-audit-s11-v2/runs/audit-fixed-s7-9cb359cb30d4/audits.jsonl",
            "output/native-acceptance-20261008/common-panel-audit-s12/runs/audit-fixed-s7-4bd283753162/audits.jsonl",
            "output/native-acceptance-20261008/common-panel-audit-s13/runs/audit-fixed-s7-7f453410e4dd/audits.jsonl",
            "output/native-acceptance-20261008/common-panel-audit-s14/runs/audit-fixed-s7-e66d778ffab7/audits.jsonl",
        ],
        "panel_sources": [
            "output/native-acceptance-20261008/common-panel-audit-v21/runs/audit-fixed-s7-7c5c99a761f0/audits.jsonl",
            "output/native-acceptance-20261008/common-panel-audit-v22/runs/audit-fixed-s7-dc756d1ae80e/audits.jsonl",
        ],
        "collection_configs": {
            "collect-round_robin-s11-653e5cf8df02": "output/native-acceptance-20261008/common-panel-s11/runs/collect-round_robin-s11-653e5cf8df02/config.json",
            "collect-round_robin-s12-9c0dc9e658b0": "output/native-acceptance-20261008/common-panel-s12/runs/collect-round_robin-s12-9c0dc9e658b0/config.json",
            "collect-round_robin-s13-86c899c5752a": "output/native-acceptance-20261008/common-panel-s13/runs/collect-round_robin-s13-86c899c5752a/config.json",
            "collect-round_robin-s14-ed1415a9c01d": "output/native-acceptance-20261008/common-panel-s14/runs/collect-round_robin-s14-ed1415a9c01d/config.json",
            "collect-round_robin-s21-25abe8e44217": "output/native-acceptance-20261008/common-panel-v21/runs/collect-round_robin-s21-25abe8e44217/config.json",
            "collect-round_robin-s22-f195078d2b76": "output/native-acceptance-20261008/common-panel-v22/runs/collect-round_robin-s22-f195078d2b76/config.json",
        },
    },
    "budget_reconciliation": {
        "status": "monetary_cap_removed_by_user_history_unclosed", "approved_new_paid_usd": None,
        "monetary_cap_enforced": False, "historical_total_budget_usd": 100.0,
        "authorization": "explicit_user_removed_budget_cap_2026-10-09",
        "actual_historical_usd": None, "remaining_usd": None,
        "ledger_path": "output/first-acceptance-v1-20261009/historical-budget-ledger.json",
        # 未闭合的库存不能冒充验收调用账本。 / Inventory is not a verified call ledger.
        "ledger_manifest": None, "ledger_schema_status": "historical_inventory_not_acceptance_call_rows",
        "historical_inventory_manifest": {
            "path": "output/first-acceptance-v1-20261009/historical-budget-ledger.json",
            "role": "unreconciled_inventory_not_call_ledger",
        },
        "audit_manifest": {
            "path": "output/first-acceptance-v1-20261009/resume-20261009-budget-audit/audit.json",
            "role": "historical_readonly_audit_not_final_billing",
        },
        "observed_workspace_gross_usd": 1.81065343,
        "paid_execution_blocked": True,
        "paid_execution_blocked_reason": "unverified_interactions_and_explicit_stage_approval_not_monetary_cap",
    },
    "stop_conditions": ["unverified_cumulative_interactions", "unverified_data_isolation",
        "interaction_limit_reached", "tuning_config_limit_reached", "no_evidence_supported_candidate",
        "ineffective_or_incomplete_attack_evidence", "confirmation_complete_no_further_tuning"],
    "research_stop": {
        "active": True, "scope": "unverified_interaction_training_or_untrusted_privacy_claims",
        "conditions": ["unverified_cumulative_interactions", "ineffective_or_incomplete_attack_evidence"],
        "permit_new_training": False, "permit_new_audits": False,
        "permit_new_calibration_search": True, "permit_attack_validity_precheck": True,
        "decision_path": "output/first-acceptance-v1-20261009/nohash-contract-stop-20261009/stop-decision.json",
    },
    "sample_seed": 7, "seeds": [101, 102, 103, 104, 105],
    "sample_action": 0, "model_rounds": [0, 12],
    "iterations": 1000, "restarts": 3, "ratio": 0.95,
    "sample_timeout_seconds": 600, "preflight_timeout_seconds": 120,
    "controller_timeout_seconds": 900, "max_containers": 1,
    "policy_sample_timeout_seconds": 300, "policy_sample_interactions": 500,
    "sample_budget_usd": None, "total_budget_usd": None,
    "estimated_l4_container_usd_per_second": 0.0003,
    # L4 估计仅解释历史；H100 按当前公开价格重新预留，不是账单硬上限。 / Historical L4 vs future H100.
    "estimated_container_usd_per_second": 0.0016,
    "hardware_benchmark": {
        "status": "hardware_probe_verified_formal_budget_unreconciled", "future_runs_only": True,
        "verification_artifact": "output/first-acceptance-v1-20261009/modal-h100-hardware-probe-v1/verified-result.json",
        "sample_timeout_seconds": 120, "controller_timeout_seconds": 180,
        "price_checked_on": "2026-10-09", "price_source": "https://modal.com/pricing",
        "published_base_compute_usd_per_second": 0.00144868,
        "speedup_verified": False, "allow_gpu_fallback": False,
        "probe": {
            "app_name": "cloud-edge-rl-h100-hardware-probe-20261009",
            "environment": "main", "profile": "oktinner0320",
            "seed": 9101, "fedsgd_rounds": 2,
            "startup_timeout_seconds": 45, "result_timeout_seconds": 90,
            "approved_sample_usd": 0.50,
            "authorization": "explicit_user_hardware_test_only_2026-10-09",
            "diagnostic_only": True, "formal_training_enabled": False,
            "output_root": "output/first-acceptance-v1-20261009/modal-h100-hardware-probe-v1",
        },
    },
    "absolute_tolerance_zero_reference": None,
    "collection_seeds": [11, 12, 13, 14],
    "collection_timeout_seconds": 120,
    "validation_timeout_seconds": 120,
    "validation_startup_wait_seconds": 300,
    "policy_validation_seeds": {"101": 201, "102": 202, "103": 203, "104": 204, "105": 205},
    "partitions": ["iid"], "diagnostic_partitions": ["label_skew"],
    "scheduler_methods": ["dqn", "ppo", "pqn", "metacs", "fixed", "time_greedy", "risk_greedy"],
    "max_tuning_configs": 9,
    "attack_candidates": {
        "categorical-softmax-v1": {"input_parameterization": "categorical_softmax", "categorical_temperature": 1.0},
        "categorical-softmax-vectorized-v1": {"input_parameterization": "categorical_softmax", "categorical_temperature": 1.0, "restart_execution": "vectorized_analytic"},
        "categorical-softmax-compiled-v1": {"input_parameterization": "categorical_softmax", "categorical_temperature": 1.0, "restart_execution": "vectorized_analytic", "compile_gradient_objective": True},
        "categorical-softmax-cudagraph-v1": {"input_parameterization": "categorical_softmax", "categorical_temperature": 1.0, "restart_execution": "vectorized_analytic", "compile_gradient_objective": True, "compile_mode": "reduce-overhead"},
        "public-gradient-prior-v1": {"input_parameterization": "categorical_softmax", "categorical_temperature": 1.0, "restart_execution": "vectorized_analytic", "compile_gradient_objective": True, "compile_mode": "reduce-overhead", "initialization": "public_gradient_prior"},
    },
    "attack_sample_profiles": {
        "strong-softmax-v1": {"attack_candidate": "categorical-softmax-v1", "iterations": 10000, "learning_rate": 0.06, "timeout_seconds": 1200},
        "strong-softmax-vectorized-v1": {"attack_candidate": "categorical-softmax-vectorized-v1", "iterations": 10000, "learning_rate": 0.06, "timeout_seconds": 1200},
        "strong-softmax-compiled-v1": {"attack_candidate": "categorical-softmax-compiled-v1", "iterations": 10000, "learning_rate": 0.06, "timeout_seconds": 1200},
        "strong-softmax-cudagraph-v1": {"attack_candidate": "categorical-softmax-cudagraph-v1", "iterations": 10000, "learning_rate": 0.06, "timeout_seconds": 1200},
        "strong-public-gradient-prior-v1": {"attack_candidate": "public-gradient-prior-v1", "iterations": 10000, "learning_rate": 0.06, "timeout_seconds": 1200},
    },
    "risk_candidates": {
        "epochs-100": {"epochs": 100},
        "epochs-100-b16": {"epochs": 100, "batch_size": 16},
        "width64-100-b16": {"epochs": 100, "batch_size": 16, "hidden_sizes": [64, 64]},
        "width64-k3-100-b16": {"epochs": 100, "batch_size": 16, "hidden_sizes": [64, 64], "ensemble_size": 3},
        "width64-100-b16-mean-calibrated": {"epochs": 100, "batch_size": 16, "hidden_sizes": [64, 64], "mean_calibration": True},
        "width64-100-b16-batch-calibrated": {"epochs": 100, "batch_size": 16, "hidden_sizes": [64, 64], "mean_calibration": True, "calibration_feature_index": 1},
        "width64-100-b16-batch-nonnegative": {"epochs": 100, "batch_size": 16, "hidden_sizes": [64, 64], "mean_calibration": True, "calibration_feature_index": 1, "calibration_nonnegative": True},
    },
    "policy_candidates": {
        "public-budget-scaling-v1": {"observation_divisors": {
            "history": [CONFIG["environment"]["max_rounds"], CONFIG["environment"]["sample_budget"], CONFIG["environment"]["sample_budget"]],
            "risk_ledger": CONFIG["environment"]["sample_budget"],
            "resources": [[up * (1 + CONFIG["resources"]["bandwidth_jitter"]), down * (1 + CONFIG["resources"]["bandwidth_jitter"]), 1.0]
                          for up, down in zip(CONFIG["resources"]["bandwidth_mbps"], CONFIG["resources"]["download_mbps"])],
            "candidates": [1.0, CONFIG["reward"]["time_scale"], CONFIG["reward"]["energy_scale"], 1.0, 1.0],
        }},
    },
    # 开发诊断盈亏平衡约 3.75；只在新训练预算中试验。 / Development-only feasibility tuning.
    "reward_candidates": {"accuracy-feasibility-v1": {"accuracy_weight": 4.0}},
    "backend_numerical_tolerance": 1e-6,
    "accuracy_slack": 0.01, "cost_multiplier": 1.15,
    "budget_by_stage_usd": {"baseline": None, "risk": None, "policy": None, "formal": None},
    "shock": {"start_seconds": 2.0, "end_seconds": 8.0, "client_ids": [0],
              "bandwidth_factor": 0.25, "load_increment": 0.3},
    "collection_validation_seeds": [21, 22],
    "audit_shard_calls": 12, "audit_shard_seconds": 480,
    "audit_timeout_seconds": 600, "fit_timeout_seconds": 180,
}
# PPO 小预算 KL 约 1e-4：下一候选仅改学习率，不改奖励或数据。 / One-variable bounded PPO trial.
CONFIG["native_acceptance"]["policy_candidates"]["scaled-ppo-lr3e3-v1"] = {
    **CONFIG["native_acceptance"]["policy_candidates"]["public-budget-scaling-v1"],
    "learning_rate": 0.003,
}
# R2 只改变偏移符号；原 R1 配置与产物保留。 / R2 changes correction sign only.
CONFIG["native_acceptance"]["development_history"] = {
    "R1-raw-vs-batch": CONFIG["native_acceptance"]["development"],
}
CONFIG["native_acceptance"]["development"] = {
    "hypothesis_id": "R2-batch-positive-part", "diagnostic_only": True,
    "status": "preregistered_before_positive_part_results",
    "change": "correction_max_zero_only_same_signed_fit",
    "risk_candidate": "width64-100-b16-batch-nonnegative",
    "base_risk_candidate": "width64-100-b16-batch-calibrated",
    "risk_fit_seeds": [501, 502, 503], "panel_seeds": [31, 32, 33],
    "shared_fit_source_root": "output/first-acceptance-v1-20261009/risk-R1-raw-vs-batch-cpu-v1",
    "training_sources": CONFIG["native_acceptance"]["development_history"]["R1-raw-vs-batch"]["training_sources"],
    "collection_configs": {key: value for key, value in CONFIG["native_acceptance"]["development_history"]["R1-raw-vs-batch"]["collection_configs"].items() if "-s21-" not in key and "-s22-" not in key},
    "panel_sources": [], "prohibit_old_panel_reads": True,
    "allow_new_fedsgd_interactions": False, "allow_new_audit_calls": False,
    "device": "cpu", "threads": 2, "precheck_seed": 501,
    "preflight_timeout_seconds": 90, "mechanism_timeout_seconds": 90,
    "new_interaction_limit": 0, "new_audit_call_limit": 0,
    "output_root": "output/first-acceptance-v1-20261009/risk-R2-positive-part-cpu-v1",
    "decision_rule": "mechanism_precheck_then_fresh_development_panel_or_reject_no_retuning",
}
# R2 已因训练数据样本门 EWMAE 比值 1.1515269286100738 被拒绝，不扩样。
CONFIG["native_acceptance"]["development_history"]["R2-batch-positive-part"] = CONFIG["native_acceptance"]["development"]
CONFIG["native_acceptance"]["development_history"]["R2-batch-positive-part"]["status"] = "rejected_training_precheck_ewmae_no_expansion"
CONFIG["native_acceptance"]["risk_candidates"]["width64-100-b16-batch-positive-half"] = {
    **CONFIG["native_acceptance"]["risk_candidates"]["width64-100-b16-batch-nonnegative"],
    "calibration_strength": 0.5,
}
CONFIG["native_acceptance"]["development"] = {
    **CONFIG["native_acceptance"]["development_history"]["R2-batch-positive-part"],
    "hypothesis_id": "R3-positive-half-postclip",
    "status": "preregistered_before_half_step_results",
    "change": "half_interpolation_after_clip_only_same_signed_fit",
    "risk_candidate": "width64-100-b16-batch-positive-half",
    "full_step_candidate": "width64-100-b16-batch-nonnegative",
    "prediction_absolute_tolerance": 5e-7, "metric_envelope_tolerance": 5e-7,
    "output_root": "output/first-acceptance-v1-20261009/risk-R3-positive-half-cpu-v1",
    "decision_rule": "fixed_half_no_strength_sweep_training_gate_then_unused_development_panel",
}
# 运行状态登记不改实验参数；原事前快照保留在 R3 registration.json。
CONFIG["native_acceptance"]["development"]["status"] = "training_precheck_passed_independent_panel_blocked"
CONFIG["native_acceptance"]["research_stop"]["decision_path"] = "output/first-acceptance-v1-20261009/risk-R3-positive-half-cpu-v1/resume-state.json"
# 本地准入缺口归集，不是新算法配置、训练或攻击。 / Offline admission evidence only.
CONFIG["native_acceptance"]["admission_preflight"] = {
    "id": "seed31-json-attribution-v1",
    "output_root": "output/first-acceptance-v1-20261009/resume-seed31-admission-20261009-v1",
    "target_method": "round_robin", "target_seed": 31, "target_split": "validation",
    "purpose": "R3_fixed_half_candidate_development_panel_admission_only",
    "max_native_manifests": 80, "max_critical_events": 12, "max_auxiliary_json": 16,
    "max_file_bytes": 8388608, "max_total_read_bytes": 67108864,
    "timeout_seconds": 90, "new_cloud_calls": 0, "new_interactions": 0,
    "probe_record": "output/first-acceptance-v1-20261009/resume-20261009-budget-audit/audit.json",
    "stop_rule": "stop_after_critical_sample_if_required_attribution_is_unresolved_no_full_manifest_crawl",
    "same_validation_partition_is_not_itself_group_overlap": True,
}
CONFIG["native_acceptance"]["admission_preflight"]["status"] = "critical_sample_stopped_no_full_expansion"
CONFIG["native_acceptance"]["research_stop"]["decision_path"] = "output/first-acceptance-v1-20261009/resume-seed31-admission-20261009-v1/resume-state.json"
# 仅补原 probe 证据；不得把原生全表扩样或新算法试验混入。
CONFIG["native_acceptance"]["admission_followup"] = {
    "id": "original-probe-and-global-quota-v2",
    "output_root": "output/first-acceptance-v1-20261009/resume-original-probe-quota-20261009-v2",
    "probe_root": "output/first-acceptance-v1-20261009/modal-h100-hardware-probe-v1",
    "probe_run_id": "collect-round_robin-s9101-ba19794136ab",
    "max_json_files": 24, "max_file_bytes": 8388608, "max_total_read_bytes": 67108864,
    "timeout_seconds": 90, "new_cloud_calls": 0, "new_interactions": 0,
    "stop_rule": "no_paid_execution_until_global_configuration_and_group_quota_closed",
    "configuration_scope": "global_including_actual_policy_and_risk_tuning_not_registration_count",
    "configuration_count_rule": "user_confirmed_global_per_component_actual_settings",
    "seeds_retraining_resume_and_fixed_denominators_not_new_configurations": True,
    "quota_stop_rule": "no_new_settings_at_or_above_native_max_fixed_evaluation_requires_other_gates",
    "policy_manifest_witnesses": [
        "output/native-acceptance-20261008/policy-pilot-dqn-s101/manifest.json",
        "output/native-acceptance-20261008/policy-scaled-dqn-s102/manifest.json",
        "output/native-acceptance-20261008/policy-accuracy4-dqn-s103-sample/manifest.json",
        "output/native-acceptance-20261008/policy-ppo-lr3e3-s104-sample-v2/manifest.json",
    ],
    "risk_manifest_witnesses": [
        "output/native-acceptance-20261008/risk-pilot3-native/manifest.json",
        "output/native-acceptance-20261008/risk-fourgroup-native/manifest.json",
        "output/native-acceptance-20261008/risk-calibrated-fourgroup-native-v4/manifest.json",
    ],
}
# 原八组上限的超额记录保留；用户仅修订至九组，不授权新配置或读取官方测试。
CONFIG["native_acceptance"]["admission_followup"]["status"] = "historical_original_cap_8_exceeded_current_cap_9_amended"
CONFIG["native_acceptance"]["research_stop"].update({
    "scope": "new_configuration_search_and_unverified_execution",
    "conditions": ["unverified_cumulative_interactions", "unverified_data_isolation", "ineffective_or_incomplete_attack_evidence", "tuning_config_limit_reached"],
    "permit_new_calibration_search": False, "permit_attack_validity_precheck": False,
    "decision_path": "output/first-acceptance-v1-20261009/quota-amendment-nine-20261009-v3/amended-state.json",
})
CONFIG["native_acceptance"]["budget_reconciliation"]["paid_execution_blocked_reason"] = "unverified_interactions_and_data_isolation_no_stage_approval"
CONFIG["native_acceptance"]["protocol_amendment"] = {
    "id": "quota-eight-to-nine-user-20261009",
    "authorization": "explicit_user_amendment_to_nine_preserve_formal_test",
    "status": "quota_revision_only_not_experiment_approval",
    "historical_max_tuning_configs": 8, "verified_actual_configuration_lower_bound": 9,
    "exact_actual_configuration_count_closed": False, "verified_new_configuration_slots": 0,
    "allow_additional_tuning": False, "allow_premature_official_test": False,
    "development_evaluation_policy": "fixed_existing_candidate_after_usage_isolation_and_bounded_stage_approval",
    "original_eight_limit_breach_preserved": True,
    "output_root": "output/first-acceptance-v1-20261009/quota-amendment-nine-20261009-v3",
}
# 用户授权：保留至少十组历史超额，禁止新增调参，仅推进已有固定候选独立开发验证。
CONFIG["native_acceptance"]["fixed_validation"] = {
    "id": "R3-independent-development-v4",
    "output_root": "output/first-acceptance-v1-20261009/independent-development-validation-20261009-v4",
    "status": "stopped_audit_submission_contract_failure_no_automatic_retry",
    "stop_decision_path": "output/first-acceptance-v1-20261009/independent-development-validation-20261009-v4/audit-s31-precheck-stage-v2/stop-state.json",
    "authorization": "explicit_user_fixed_validation_despite_retained_historical_configuration_overshoot",
    "historical_configuration_lower_bound": 10, "historical_configuration_total": None,
    "historical_overshoot_blocks_fixed_evaluation": False, "allow_new_tuning": False,
    "candidate": "width64-100-b16-batch-positive-half", "risk_fit_seeds": [501, 502, 503],
    "panel_method": "round_robin", "panel_seeds": [31, 32, 33], "panel_split": "validation",
    "collection_profile": {"source_manifest": "output/native-acceptance-20261008/common-panel-s11/manifest.json", "round_robin_actions": [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11], "risk_mode": "unavailable"},
    "modal_environment_name": "main",
    "official_test_allowed": False, "allow_refit_or_parameter_changes": False,
    "admission_timeout_seconds": 90, "max_json_files": 120,
    "max_file_bytes": 8388608, "max_total_read_bytes": 67108864,
    "sample_native_manifests": 12, "max_native_manifests": 80,
    "training_shard_interactions": 100, "training_shard_timeout_seconds": 900,
    "audit_shard_uploads": 12, "audit_shard_timeout_seconds": 1200,
    "max_containers": 1, "paid_stage_approved": False,
    "approved_scope": "collection_seed31_only_no_attacks_no_policy_training",
    "stage_dir": "output/first-acceptance-v1-20261009/independent-development-validation-20261009-v4/collect-s31-stage-v2",
    "collection_startup_timeout_seconds": 180,
    "scoped_historical_ledger_path": "output/first-acceptance-v1-20261009/independent-development-validation-20261009-v4/collect-s31-stage/scoped-historical-ledger.json",
    "max_new_interactions_this_stage": 30,
    "source_identity": "exact_content_snapshot_no_hash",
    "approval_max_file_bytes": 33554432, "approval_max_total_read_bytes": 134217728,
    "audit_profile": {
        "id": "R3-matched-training-attack-development-only",
        "prototype_manifest": "output/native-acceptance-20261008/common-panel-audit-s11-v2/manifest.json",
        "attack": {**CONFIG["attack"], "iterations": CONFIG["native_acceptance"]["iterations"], "restarts": CONFIG["native_acceptance"]["restarts"]},
        "backend": CONFIG["components"]["backend"], "attack_component": "tabular_gia", "attack_seed": CONFIG["native_acceptance"]["sample_seed"],
        "comparisons_attack": CONFIG["comparisons"]["attack"],
        "execution_backend": "modal",
        "selection": "earliest_four_postwarm_uploads_per_batch_size_before_labels",
        "batch_sizes": [8, 32, 64], "per_batch_uploads": 4,
        "precheck_uploads": 3, "precheck_timeout_seconds": 900, "precheck_attack_seconds": 480,
        "remainder_uploads": 9, "max_panel_uploads": 12,
        "new_interactions": 0, "new_fits": 0, "formal_attack_strength_claim": False,
        "paid_precheck_approved": False, "paid_remainder_approved": False,
        "approval_scope": "stopped_after_bound_call_failure_requires_fresh_bounded_approval",
        "selection_plan_path": "output/first-acceptance-v1-20261009/independent-development-validation-20261009-v4/audit-s31-precheck-stage/selection-plan.json",
        "precheck_stage_dir": "output/first-acceptance-v1-20261009/independent-development-validation-20261009-v4/audit-s31-precheck-stage-v2",
        "stop_if": ["timeout_or_failure", "truth_leakage", "invalid_nonfinite_labels", "no_attack_diagnostics"],
        "expand_only_after": "completed_three_batch_precheck_with_valid_labels_and_primary_approval_no_automatic_retry",
    },
    "stop_rule": "stop_if_relevant_counters_dependencies_or_isolation_unresolved_no_automatic_retry",
    "retain_failures_and_all_denominators": True,
}
# 本轮目标改为交付首轮普通真实开发结果，不要求性能达标；正式确认暂缓。
CONFIG["native_acceptance"]["ordinary_results"] = {
    "id": "ordinary-development-first-results-v1",
    "output_root": "output/first-acceptance-v1-20261009/independent-development-validation-20261009-v4/ordinary-results-20261009-v1",
    "reuse_training_seed": 101, "reuse_validation_seed": 201,
    "methods": ["dqn", "ppo", "pqn", "fixed", "time_greedy", "risk_greedy", "metacs"],
    "independent_panel_seeds": [31], "risk_fit_seeds": [501, 502, 503],
    "no_new_tuning": True, "no_new_training_or_collection": True,
    "official_test_allowed": False, "formal_confirmation_deferred": True,
    "save_failed_numerical_results": True, "stop_after_results_and_analysis": True,
    "maximum_new_attack_calls": 12, "maximum_new_compute_calls": 2,
    "preflight_timeout_seconds": 90, "precheck_timeout_seconds": 900,
    "remainder_timeout_seconds": 1200, "download_timeout_seconds": 180,
    "evaluation_timeout_seconds": 90, "max_containers": 1,
    "new_interactions": 0, "new_fits": 0,
    "stop_affected_path_if": ["execution_failure", "truth_leakage", "invalid_artifacts", "unresolved_relevant_interactions", "ineffective_attack"],
    "own_true_risk_requires_complete_audit": True,
    "historical_cost_unknown_not_zero": True,
}
CONFIG["native_acceptance"]["fixed_validation"]["status"] = "ordinary_results_sample_approved_after_minimal_backend_fix"
CONFIG["native_acceptance"]["fixed_validation"]["audit_profile"]["paid_precheck_approved"] = True
CONFIG["native_acceptance"]["fixed_validation"]["audit_profile"]["approval_scope"] = "seed31_three_upload_label_precheck_only"
CONFIG["native_acceptance"]["fixed_validation"]["audit_profile"]["precheck_stage_dir"] = "output/first-acceptance-v1-20261009/independent-development-validation-20261009-v4/ordinary-results-20261009-v1/precheck-stage"
# 三份实际标签结构门通过；强度未认证，九份余量只用于固定协议开发诊断。
CONFIG["native_acceptance"]["fixed_validation"]["status"] = "ordinary_results_precheck_completed_remaining_nine_approved"
CONFIG["native_acceptance"]["fixed_validation"]["audit_profile"]["paid_remainder_approved"] = True
CONFIG["native_acceptance"]["fixed_validation"]["audit_profile"]["approval_scope"] = "seed31_remaining_nine_upload_labels_only"
# 首轮只得到 8/12：用尽两次调用后停止；修正未来余量选点入口，但不授权补跑。
CONFIG["native_acceptance"]["fixed_validation"]["audit_profile"]["remainder_policy"] = "panel"
CONFIG["native_acceptance"]["fixed_validation"]["status"] = "ordinary_results_delivered_partial_panel_stopped_waiting_user"
CONFIG["native_acceptance"]["fixed_validation"]["audit_profile"]["paid_precheck_approved"] = False
CONFIG["native_acceptance"]["fixed_validation"]["audit_profile"]["paid_remainder_approved"] = False
CONFIG["native_acceptance"]["fixed_validation"]["audit_profile"]["approval_scope"] = "stopped_after_ordinary_results_no_third_compute_call"
# 用户只批准一个新策略名额的本地准备；不启用真实训练或付费。 / Local preparation only.
CONFIG["native_acceptance"]["constraint_ppo"] = {
    "id": "ppo-resource-feedback-v1", "status": "local_implementation_verified_waiting_separate_real_execution_approval",
    "hypothesis": "stage_frozen_observed_resource_feedback_improves_joint_time_energy_feasibility",
    "approved_additional_policy_slots": 1, "approval_scope": "local_implementation_synthetic_regressions_and_registration_only",
    "historical_configuration_lower_bound": 10, "historical_exact_configuration_count": None,
    "actual_new_tuning_executions": 0, "new_cloud_budget_usd": 0.0,
    "new_real_interaction_limit": 0, "new_cloud_call_limit": 0, "validation_timeout_seconds": 90,
    "paid_stage_approved": False, "actual_training_approved": False, "official_test_allowed": False,
    "resource_feedback": {"mode": "dual", "stage_episodes": 8, "dual_step_size": 1.0,
                          "time_limit": None, "energy_limit": None},
    "control_mode": "fixed", "shared_stage_flush": True, "control_protocol_counting_requires_review_before_execution": True,
    "target_multiplier": 1.15, "frozen_scheduler_reference": None, "frozen_risk_version": None,
    "same_risk_and_calibration_in_both_arms": True, "allow_r3_strength_tuning": False,
    "canonical_pair_interaction_limit": 3000, "pair_limit_is_combined_not_per_arm": True,
    "developer_seed_and_counter_admission": "pending", "formal_confirmation_allowed": False,
    "artifact_dir": "output/first-acceptance-v1-20261009/constraint-ppo-local-20261009-v1",
}
# 用户补批两组实际策略设置：预检通过后同预算续训；全部历史和停止记录保留。 / Bounded paired approval.
CONFIG["native_acceptance"]["constraint_ppo"].update({
    "status": "stopped_fixed_continuation_readonly_download_timeout_no_auto_retry",
    "approved_additional_policy_slots": 2,
    "approval_scope": "closed_after_new_download_timeout_fresh_scoped_approval_required",
    "paid_stage_approved": False, "actual_training_approved": False,
    "new_cloud_budget_usd": 6.0, "budget_bound_kind": "estimated_reservation_not_bill_or_hard_charge_cap",
    "new_real_interaction_limit": 2400, "new_cloud_call_limit": 4,
    "developer_seed": 1201, "total_steps_per_arm": 1200, "precheck_interactions_per_arm": 30,
    "remaining_train_interactions_per_arm": 1170, "shared_allocation_max": 600,
    "precheck_timeout_seconds": 180, "continuation_timeout_seconds": 900, "startup_timeout_seconds": 180,
    "max_containers": 1, "gpu": "H100", "cpu": 16.0, "memory_mb": 65536,
    "control_protocol_counting_requires_review_before_execution": False,
    "configuration_ids_by_mode": {"fixed": "constraint-ppo-matched-fixed-v1", "dual": "constraint-ppo-resource-dual-v1"},
    "policy_candidate": "scaled-ppo-lr3e3-v1", "reward_candidate": "accuracy-feasibility-v1",
    "frozen_risk_source": "output/first-acceptance-v1-20261009/risk-R1-raw-vs-batch-cpu-v1/fit-s501/risk.pt",
    "frozen_risk_version": "risk-r3-positive-half-fit501-constraint-pair-v1",
    "frozen_risk_volume_path": "inputs/constraint-ppo-paired-20261009-v1/risk-fit501-positive-half.pt",
    "resource_anchor_scope": "frozen_development_resource_anchor_not_formal_reference",
    "resource_anchor_method": "time_greedy", "resource_anchor_training_seed": 101, "resource_anchor_validation_seed": 201,
    "resource_anchor_source": "output/first-acceptance-v1-20261009/independent-development-validation-20261009-v4/ordinary-results-20261009-v1/ordinary-results.json",
    "resource_feedback": {"mode": "dual", "stage_episodes": 8, "dual_step_size": 1.0,
                          "time_limit": 4.36313614867802, "energy_limit": 30.070671189069326},
    "local_preparation_artifact_dir": "output/first-acceptance-v1-20261009/constraint-ppo-local-20261009-v1",
    "artifact_dir": "output/first-acceptance-v1-20261009/constraint-ppo-paired-20261009-v1",
    "executed_policy_settings": 2, "stage_end_configuration_count_lower_bound": 12,
    "completed_compute_calls": 3, "unsubmitted_original_compute_calls": 1,
    "returned_new_real_interactions": 1230, "shared_allocated_once": 120,
    "returned_canonical_interactions": 1350, "unsubmitted_original_interactions": 1170,
    "latest_stop_state": "output/first-acceptance-v1-20261009/constraint-ppo-paired-20261009-v1/admission-shared-source-v2/latest-stop-state.json",
})
# 独立参数选择：仅研究/建设，不启动云训练，不修改原训练管线。 / Separate selection, no paid execution.
CONFIG["native_acceptance"]["parameter_selection"] = {
    "id": "dqn-ppo-parameter-selection-20261010-v1", "status": "approved_full_20_after_verified_zero_failure",
    "output_directory": "output/parameter-selection-dqn-ppo-20261010-v1/execution-v2",
    "paid_execution_approved": True, "tuning_configuration_slots_approved": 20,
    "groups_per_algorithm": 10, "environment_interactions_per_group": 500,
    "training_seed": 5101, "evaluation_seeds": [6201, 6202], "interaction_limit": 5120,
    "historical_configuration_lower_bound": 12, "historical_configuration_total": None,
    "historical_interactions": {"dqn": 0, "ppo": 0},
    "historical_direct_count_scope": "verified unused method/seed5101/train groups only; unknown global history retained",
    "allocated_collection_interactions": {"dqn": 120, "ppo": 120},
    "frozen_risk_checkpoint": "inputs/constraint-ppo-paired-20261009-v1/risk-fit501-positive-half.pt",
    "frozen_risk_local_checkpoint": "output/first-acceptance-v1-20261009/constraint-ppo-paired-20261009-v1/frozen-risk.pt",
    "frozen_risk_version": "risk-r3-positive-half-fit501-constraint-pair-v1",
    "frozen_risk_config": {"mode": "unavailable", "ensemble_size": 5, "hidden_sizes": [64, 64],
        "learning_rate": 0.001, "epochs": 100, "batch_size": 16, "device": "cpu",
        "validation_group_fraction": 0.2, "validation_fraction": 0.2, "weight_decay": 0.0001,
        "mean_calibration": True, "calibration_feature_index": 1, "seed": 501,
        "calibration_nonnegative": True, "calibration_strength": 0.5},
    "registered_spec_paths": [], "official_test_allowed": False,
    "scoped_quota_amendment": {"approved": True, "scope_id": "dqn-ppo-parameter-selection-20261010-v1",
        "additional_configuration_slots": 20, "training_interactions_by_algorithm": {"dqn": 5000, "ppo": 5000},
        "cumulative_interaction_limit": 5120, "historical_unknowns_retained": True, "official_test_access": False,
        "monetary_cap_waived_by_user": True, "historical_actual_cloud_cost_usd": None,
        "max_development_validation_interactions": 1260, "maximum_compute_calls": 62,
        "scope": "this frozen twenty-configuration development batch only; other quotas unchanged"},
    "shared_execution_descriptors": {
        "source": "output/parameter-selection-dqn-ppo-20261010-v1/execution-v2/source-descriptor.json",
        "admission": "output/parameter-selection-dqn-ppo-20261010-v1/execution-v2/admission-descriptor.json"},
    "reservation_log": "output/parameter-selection-dqn-ppo-20261010-v1/execution-v1/reservations.jsonl",
    "retained_failed_execution": {"run_id": "train_policy-dqn-D00-s5101-9c46b2ac735a42e5bcdba057b58856ca",
        "directory": "output/parameter-selection-dqn-ppo-20261010-v1/execution-v1/failed-D00-recovery-20261010-v1",
        "new_user_approval_scope": "Complete all 20 registered candidates and all paired development validation; preserve failed call and costs, official test sealed"},
    "stage_execution": {"max_new_jobs": 3, "watchdog_seconds": 5400, "automatic_retry": False,
        "freeze_preflight_seconds": 90, "max_json_files": 120, "max_json_file_bytes": 67108864, "max_json_total_bytes": 134217728},
    "baseline_rl": {
        "hidden_sizes": [128, 128], "learning_rate": 0.0003, "gamma": 1.0,
        "batch_size": 64, "buffer_size": 20000, "learning_starts": 256,
        "train_frequency": 1, "target_update_interval": 100,
        "epsilon_start": 1.0, "epsilon_end": 0.05, "epsilon_decay_steps": 2000,
        "max_grad_norm": 10.0, "double_dqn": True,
        "rollout_steps": 128, "ppo_epochs": 4, "clip_range": 0.2,
        "gae_lambda": 0.95, "entropy_coefficient": 0.01,
        "value_coefficient": 0.5, "normalize_advantage": True,
        "ppo": {"rollout_steps": 128, "epochs": 4, "clip_ratio": 0.2,
                "gae_lambda": 0.95, "entropy_weight": 0.01},
    },
    "baseline_reward": {"protocol": "exposure_v1", "accuracy_weight": 1.0,
        "time_weight": 0.05, "energy_weight": 0.01, "risk_weight": 0.1,
        "uncertainty_weight": 0.0, "time_scale": 1.0, "energy_scale": 10.0},
    "selection": {"accuracy_slack": 0.01, "cost_multiplier": 1.15,
        "reference_method": "time_greedy",
        "ordering": ["accuracy_desc", "time_asc", "energy_asc", "configuration_id_asc"]},
    "candidates": [
        {"id": "D00", "algorithm": "dqn", "override": {}, "hypothesis": "short_budget_historical_hyperparameter_control"},
        {"id": "D01", "algorithm": "dqn", "override": {"learning_starts": 64}, "hypothesis": "earlier_updates_improve_short_budget_learning"},
        {"id": "D02", "algorithm": "dqn", "override": {"epsilon_decay_steps": 300}, "hypothesis": "faster_exploration_decay_improves_deployment"},
        {"id": "D03", "algorithm": "dqn", "override": {"target_update_interval": 20}, "hypothesis": "frequent_targets_propagate_short_budget_values"},
        {"id": "D04", "algorithm": "dqn", "override": {"batch_size": 32}, "hypothesis": "smaller_batches_improve_small_replay_learning"},
        {"id": "D05", "algorithm": "dqn", "override": {"learning_rate": 0.0001}, "hypothesis": "lower_learning_rate_limits_q_instability"},
        {"id": "D06", "algorithm": "dqn", "override": {"learning_rate": 0.001}, "hypothesis": "higher_learning_rate_accelerates_q_learning"},
        {"id": "D07", "algorithm": "dqn", "override": {"epsilon_decay_steps": 500}, "hypothesis": "intermediate_decay_balances_coverage_and_exploitation"},
        {"id": "D08", "algorithm": "dqn", "override": {"target_update_interval": 40}, "hypothesis": "intermediate_target_clock_balances_feedback_and_stability"},
        {"id": "D09", "algorithm": "dqn", "override": {"batch_size": 128}, "hypothesis": "larger_batches_reduce_q_gradient_noise"},
        {"id": "P00", "algorithm": "ppo", "override": {}, "hypothesis": "short_budget_historical_hyperparameter_control"},
        {"id": "P01", "algorithm": "ppo", "override": {"rollout_steps": 64}, "hypothesis": "shorter_rollouts_refresh_policy_more_often"},
        {"id": "P02", "algorithm": "ppo", "override": {"learning_rate": 0.001}, "hypothesis": "higher_learning_rate_accelerates_policy_adaptation"},
        {"id": "P03", "algorithm": "ppo", "override": {"clip_range": 0.1}, "hypothesis": "tighter_clip_limits_small_sample_policy_drift"},
        {"id": "P04", "algorithm": "ppo", "override": {"gae_lambda": 0.9}, "hypothesis": "shorter_gae_limits_early_critic_error_propagation"},
        {"id": "P05", "algorithm": "ppo", "override": {"learning_rate": 0.0001}, "hypothesis": "lower_learning_rate_stabilizes_policy_updates"},
        {"id": "P06", "algorithm": "ppo", "override": {"clip_range": 0.3}, "hypothesis": "wider_clip_allows_useful_local_policy_adjustment"},
        {"id": "P07", "algorithm": "ppo", "override": {"entropy_coefficient": 0.0}, "hypothesis": "no_entropy_bonus_reduces_persistent_exploration"},
        {"id": "P08", "algorithm": "ppo", "override": {"ppo_epochs": 8}, "hypothesis": "more_epochs_improve_fixed_sample_use"},
        {"id": "P09", "algorithm": "ppo", "override": {"rollout_steps": 256}, "hypothesis": "longer_rollouts_reduce_advantage_estimation_noise"},
    ],
}
CONFIG["native_acceptance"]["parameter_selection"]["registered_spec_paths"] = [
    f"output/parameter-selection-dqn-ppo-20261010-v1/execution-v2/specs/train-{row['id']}.json"
    for row in CONFIG["native_acceptance"]["parameter_selection"]["candidates"]
] + [
    f"output/parameter-selection-dqn-ppo-20261010-v1/execution-v2/specs/validate-{row['id']}-{seed}.json"
    for row in CONFIG["native_acceptance"]["parameter_selection"]["candidates"] for seed in (6201, 6202)
] + [
    f"output/parameter-selection-dqn-ppo-20261010-v1/execution-v2/specs/reference-time_greedy-{seed}.json" for seed in (6201, 6202)
]
CONFIG["modal"].update({"gpu": "H100", "cpu": 16.0, "memory_mb": 65536,
    "timeout_seconds": 600, "startup_timeout_seconds": 180, "max_containers": 1, "retries": 0})
# 参考策略使用同一真实交互预算和有限时域折扣。 / Paired actual interaction budgets.
CONFIG["comparisons"]["pqn"]["budget_includes_warmup"] = True
CONFIG["comparisons"]["pqn"]["gamma"] = 1.0
CONFIG["comparisons"]["pqn"]["total_steps"] = CONFIG["rl"]["total_steps"]
CONFIG["comparisons"]["pqn"]["epsilon_decay_steps"] = CONFIG["rl"]["epsilon_decay_steps"]
CONFIG["comparisons"]["pqn"]["lr_decay_steps"] = CONFIG["rl"]["total_steps"]
CONFIG["comparisons"]["risk"]["validation_fraction"] = CONFIG["risk"]["validation_fraction"]
# 单入口仅执行已审批冻结快照；空列表不提交、不重新生成 UUID。 / Approved snapshots only.
CONFIG["workflow"]["training_pipeline"] = {
    "registered_spec_paths": [], "output_directory": "output/training-pipeline",
}
CONFIG["modal"]["transport"] = {
    "listing_seconds": 45, "file_seconds": 45, "download_seconds": 600,
    "cleanup_seconds": 90, "verification_seconds": 60,
    "status_reads": 6, "status_interval_seconds": 2, "read_attempts": 2,
    "max_entries": 32, "max_file_bytes": 67108864, "max_total_bytes": 134217728,
}


## 2. 框架核验与数据 / Runtime and data

保留官方测试集，仅训练部分拟合编码。首次下载真实 Adult ZIP 并记录 SHA256。缺失与未知类别分别编码；数据划分不依赖策略。

重复执行已完成阶段时读取原结果；改源码或配置会生成新的运行 ID。

In [ ]:
RESULTS["runtime"] = execute_job(CONFIG, "runtime", PROJECT_ROOT)
RESULTS["data"] = execute_job(CONFIG, "data", PROJECT_ROOT)
display(pd.DataFrame(RESULTS["data"]))

## 3. Modal 连通性 / Modal connectivity

仅在配置区将 `modal_check.enabled` 设为 True 才提交。首次会构建锁定镜像，再检查真实 CUDA 可用性，不运行学习核心。调用 ID 保存到 output/submissions；云端产物回收到本机 runs。

终端一次性认证使用 `uv run modal setup`；凭证不写入 Notebook 或 Git。

In [ ]:
RESULTS["modal_check"] = execute_job(CONFIG, "modal_check", PROJECT_ROOT)
display(RESULTS["modal_check"])

## 4. 真实预实验和风险采样 / Real pilot and risk collection

先实现 `FederatedBackend.reset/step_round/checkpoint/load_checkpoint`。初始采样用显式 risk=None 模式，risk_available 为 0，审计预测为 None，不表示真实风险等于零。固定的 12 动作预热计入时间、样本与暴露预算。需要小型真实攻击前先确保至少覆盖 8/32/64 三种 batch。

后续启用 train 时，同样在用户训练循环中调用 `baselines.run_warmup`，与基线使用相同历史信息与成本。

In [ ]:
RESULTS["collect"] = execute_job(CONFIG, "collect", PROJECT_ROOT)
display(RESULTS["collect"])

## 5. 攻击成本与离线标签 / Attack cost and offline labels

实现 `GradientAttack.reconstruct` 后，把 collect 的 run_id 填入配置区 benchmark.inputs.source_run_ids。benchmark 对每种 batch 的一个已上传梯度实测耗时；再把 benchmark run_id 填入 audit.inputs.benchmark_run_id，并填写 source_run_ids。

定期/随机/主动审计使用相同调用和算时预算。已启动的单次攻击可能超出剩余估算，超出量会完整记录；主动审计不能在没有真实风险模型时使用。

In [ ]:
RESULTS["benchmark"] = execute_job(CONFIG, "benchmark", PROJECT_ROOT)
RESULTS["audit"] = execute_job(CONFIG, "audit", PROJECT_ROOT)
display(RESULTS["benchmark"] + RESULTS["audit"])

## 6. 风险拟合 / Risk fitting

填写 fit_risk.inputs.audit_run_ids，然后实现 RiskEstimator 的 fit/predict/checkpoint/load_checkpoint。输入按完整 run_id 分组，验证组划分由您在核心中完成。任何 test/validation 审计标签都不能进入此拟合入口。

输出 runs/<run_id>/risk.pt，后续训练显式引用此路径和 risk_version。

In [ ]:
RESULTS["fit_risk"] = execute_job(CONFIG, "fit_risk", PROJECT_ROOT)
display(RESULTS["fit_risk"])

## 7. DQN 主线与调参 / DQN and tuning

实现 train_agent、Agent 和 load_agent。配置区 train.inputs.risk_checkpoint 指向已完成的风险版本。普通/保守/无隐私项 DQN 通过 variants 明确区分。PPO 保留同一接口，主线完成后再启用。

云端全量训练使用 CONFIG.workflow.training_pipeline.registered_spec_paths 中已审批的冻结任务列表；一次启动按顺序推进训练、停止确认、下载与语义核验。恢复沿用原 run_id/call，不重训已完成任务；空列表或缺少审批直接拒绝。详见 docs/训练管线.md。

`tuning` 声明搜索空间；所有候选应使用相同验证种子与预算。默认只展示候选；在集中配置启用 tune 才逐候选运行相同预算训练。选择最优参数必须基于验证场景，最终 test 不参与选择。

将训练或 tune 产生的 run_id 填入 validate.inputs.training_run_ids 并启用 validate。框架从每个训练快照恢复对应超参数，在共同验证种子上评价，且不读取最终测试集。选定 trial 后在 evaluate.inputs.training_run_ids 引用其运行；框架保留其训练快照用于最终评价。 / Validation restores each trial snapshot on shared held-out seeds.

In [ ]:
from cloud_edge_rl.tuning import sample_trials

display(pd.DataFrame(sample_trials(CONFIG["tuning"])))
RESULTS["tune"] = execute_job(CONFIG, "tune", PROJECT_ROOT)
RESULTS["train"] = execute_job(CONFIG, "train", PROJECT_ROOT)
display(RESULTS["train"])
RESULTS["validate"] = execute_job(CONFIG, "validate", PROJECT_ROOT)
display(RESULTS["validate"])

## 8. 分阶段审计校准 / Stagewise calibration

将新采集的 train 审计运行加入 calibrate.inputs.audit_run_ids（包含原训练标签以免遗忘）。新版本在新阶段使用；阶段内始终冻结。用户 RL 核心需要保留策略权重、清空旧版本 replay；同版本恢复则还原优化器、RNG、replay 及 RecordingEnv.snapshot。

先在同一上传记录池比较三种审计规则，再比较闭环策略，避免把采样轨迹差异当作选点收益。

In [ ]:
RESULTS["calibrate"] = execute_job(CONFIG, "calibrate", PROJECT_ROOT)
display(RESULTS["calibrate"])

## 9. 独立最终评价 / Held-out evaluation

冻结算法、风险模型和全部调参选择；将选定 trial 的训练 run_id 填入 evaluate.inputs.training_run_ids；框架恢复其完整参数和两个 checkpoint，不用当前默认值覆盖获选设置。评估使用 test 种子，业务测试集只在回合结束后读取。上传记录标记 test；其攻击标签不能回流 fit。

将 evaluate run_id 填入 evaluate_audits.source_run_ids。独立面板可将 audit.policy 设为 panel，覆盖实际保存的动作/客户端；覆盖不足应补充独立轮询采集，不能伪造反事实梯度。

透明基线使用 evaluate_baselines 阶段，填写同一冻结风险检查点，并在验证后锁定 fixed_action。 / Transparent baselines use the same frozen risk checkpoint.

In [ ]:
RESULTS["evaluate_baselines"] = execute_job(CONFIG, "evaluate_baselines", PROJECT_ROOT)
RESULTS["evaluate"] = execute_job(CONFIG, "evaluate", PROJECT_ROOT)
RESULTS["evaluate_audits"] = execute_job(CONFIG, "evaluate_audits", PROJECT_ROOT)
display(RESULTS["evaluate"] + RESULTS["evaluate_audits"])

## 10. 结果汇总与课程材料 / Analysis and deliverables

在集中配置 analysis 中填写要比较的已完成 run_id。分别按模拟时间和累计处理量绘图；每个种子作为独立统计单位，展示平均值与离散程度。未发生风险漂移、校准无效或 DQN 与贪心接近都属于可报告结果。

报告 8–20 页（不含参考文献），Arial 12、1.5 倍行距、1 英寸页边距；展示 15 分钟。Git、复现记录与贡献说明保留完整。

In [ ]:
from cloud_edge_rl.analysis import (
    analyze_risk_runs,
    pairwise_summary,
    plot_learning_curves,
    plot_risk_calibration,
    summarize_runs,
)

if CONFIG["analysis"]["run_ids"]:
    table = summarize_runs(PROJECT_ROOT, CONFIG["analysis"]["run_ids"])
    display(table)
    display(
        pairwise_summary(
            table, CONFIG["analysis"]["metric"], CONFIG["analysis"]["reference_method"]
        )
    )
    plot_learning_curves(
        PROJECT_ROOT,
        CONFIG["analysis"]["run_ids"],
        PROJECT_ROOT / "output" / "figures",
        CONFIG["analysis"]["x_axis"],
    )
if CONFIG["analysis"]["audit_run_ids"]:
    display(
        analyze_risk_runs(
            PROJECT_ROOT, CONFIG["analysis"]["audit_run_ids"], CONFIG["analysis"]["risk_threshold"]
        )
    )
    plot_risk_calibration(
        PROJECT_ROOT, CONFIG["analysis"]["audit_run_ids"], PROJECT_ROOT / "output" / "figures"
    )

## 11. 外部算法筛选 / External algorithm screening

参数仍只在 CONFIG 单元的 reference_benchmarks 修改。下载的官方源码、独立 UV 环境与实测结果位于 reference_algorithms；各候选使用适合自身接口的筛选任务，不能把跨任务指标排名当成本项目 SOTA 结论。

Tabular-GIA 使用真实 Adult 梯度；TabM 使用 Adult 分类；ConformalOpt 使用官方 GOOGL 残差；MetaCS-FL 测资源优化；PQN 测 CartPole。OCP-Unlock+ 在原论文反馈协议下实测，其覆盖反馈假设与未审计上传不匹配。默认全部禁用，已有结果可阅读 reference_algorithms/REPORT.md。


In [ ]:
from reference_algorithms.run import run_suite

REFERENCE_RESULTS = run_suite(PROJECT_ROOT, CONFIG)
display(REFERENCE_RESULTS)

## 12. 主项目基线闭环 / Integrated baseline workflow

所有参数在 CONFIG 中设置。`comparisons.workflow.enabled` 控制有依赖关系的完整比较；各阶段生成的真实运行 ID 自动作为后续输入，快照随产物保存。默认关闭，启用后将在 Modal 运行实际 FedSGD、官方反演、MLP/TabM 风险拟合、PQN、六种调度和在线校准对比。

PQN 是串行 PyTorch/Gymnasium 移植；MetaCS 使用官方目标对 12 个动作枚举。ConformalOpt/OCP 只评价已审计上传序列，覆盖率不外推到未审计上传。OCP 的信息揭示次数不表示节省了已经完成的攻击调用。计划和验收见 `docs/BASELINE_INTEGRATION.md`。


若控制器中断，先确认旧控制器已结束，在 CONFIG 中填写 resume_manifest 并使用新 tag；原实验参数保持冻结，已记录的云调用仅重新连接。 / After confirming the old controller has stopped, set resume_manifest and a new tag in CONFIG; recorded calls are reattached without retuning.

In [ ]:
from cloud_edge_rl.comparisons.workflow import resume_comparisons, run_comparisons

# 恢复路径只在 CONFIG 设置；已付费调用不重新训练。 / Set recovery in CONFIG; paid calls are not retrained.
comparison_settings = CONFIG["comparisons"]["workflow"]
if comparison_settings["resume_manifest"]:
    COMPARISON_RESULTS = resume_comparisons(
        CONFIG, PROJECT_ROOT, PROJECT_ROOT / comparison_settings["resume_manifest"]
    )
else:
    COMPARISON_RESULTS = run_comparisons(CONFIG, PROJECT_ROOT)
display({"state": COMPARISON_RESULTS["state"], "manifest": COMPARISON_RESULTS.get("manifest")})
display(
    pd.DataFrame(
        [
            {"name": phase["name"], "state": phase["state"], "runs": len(phase["run_ids"])}
            for phase in COMPARISON_RESULTS["phases"]
        ],
        columns=["name", "state", "runs"],
    )
)
if COMPARISON_RESULTS["state"] == "completed":
    # 只展示真实最终指标，不将审计标签回流。 / Show genuine final metrics without recycling audit labels.
    phases = {phase["name"]: phase for phase in COMPARISON_RESULTS["phases"]}
    scheduler_table = summarize_runs(
        PROJECT_ROOT, phases["comparison_evaluate_schedulers"]["run_ids"]
    )
    display(
        scheduler_table[
            [
                "method",
                "seed",
                "test.accuracy",
                "validation_loss",
                "elapsed_seconds",
                "energy_joules",
                "processed_samples",
            ]
        ]
    )
    scheduler_risk = analyze_risk_runs(
        PROJECT_ROOT,
        phases["comparison_scheduler_audits"]["run_ids"],
        CONFIG["analysis"]["risk_threshold"],
    )
    display(
        scheduler_risk.merge(
            scheduler_table[["run_id", "method", "seed"]],
            left_on="source_run_id",
            right_on="run_id",
        ).drop(columns=["run_id"])
    )

## 13. 首版独立验收 / First acceptance (read-only)

以 `docs/一版验收标准.md` 为唯一判据。此单元只读取已冻结的本地证据并判定，不训练、不攻击、不提交云任务。未登记完整清单时返回证据不足；历史开发结果、两个开发面板及布尔通过标记不能替代三种子独立确认。

当前已触发历史费用/证据停止：可信历史费用及余额未知，`ledger_manifest` 保持空值，库存和最新核对仅作为历史证据读取，不使用文件哈希验证。正式验收描述符采用绝对路径与完整 JSON 快照，并以非内容派生的 `freeze_id` 绑定冻结版本。R1 开发均值通过不表示正式确认通过；付费和冻结开关继续关闭，不新增训练、审计或校准搜索。

In [ ]:
import json
from cloud_edge_rl.comparisons.acceptance import first_acceptance_report

first_settings = CONFIG["native_acceptance"]
first_manifest = first_settings["confirmation"]["evidence_manifest"]
first_evidence = {}
if first_settings["enabled"] and isinstance(first_manifest, dict) and first_manifest.get("path"):
    try:
        first_evidence = json.loads((PROJECT_ROOT / first_manifest["path"]).read_text(encoding="utf-8"))
    except (OSError, ValueError) as error:
        first_evidence = {"load_error": str(error)}
FIRST_ACCEPTANCE = first_acceptance_report(first_settings, first_evidence)
FIRST_ACCEPTANCE["research_stop"] = first_settings["research_stop"]
FIRST_ACCEPTANCE["budget_reconciliation"] = {key: first_settings["budget_reconciliation"][key]
    for key in ("status", "actual_historical_usd", "remaining_usd", "approved_new_paid_usd", "paid_execution_blocked")}
RESULTS["first_acceptance"] = FIRST_ACCEPTANCE
display({key: FIRST_ACCEPTANCE[key] for key in ("status", "reasons", "checks", "research_stop", "budget_reconciliation")})